# dim_clientes
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Gold (nivel 2) · dimensión Clientes (SCD2) |
| Propósito | Una fila por versión de cliente, con su segmento y vigencia (vigente_desde, vigente_hasta, es_vigente). |
| Fuentes | silver.clientes_historial (segmento versionado) + silver.clientes (datos actuales) |
| Decisiones | - La primera versión vale desde 1900: una venta anterior al primer cambio registrado también encuentra su versión. - es_posible_duplicado: misma persona con otra cuenta (mismo email canónico, o mismo nombre + apellido + teléfono). Se marca; fusionar exige una regla de identidad del negocio. - Datos personales: nombre, apellido y email llevan máscara de columna de Unity Catalog (funciones en <catalogo>.gobierno); solo grp-andina-pii-lectores ve el dato real. |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.1 · 2026-10-14 · máscaras de columna sobre nombre, apellido y email |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import Window
from pyspark.sql import functions as F

from comun_gold import CATALOGO, FIN_DE_LOS_TIEMPOS, INICIO_DE_LOS_TIEMPOS, leer_silver, sk

# Funciones de máscara de datos personales (infra/04_unity_catalog_objetos.sql).
GOBIERNO = f"{CATALOGO}.gobierno"   # andina_dev.gobierno en dev, andina_prd.gobierno en prd

## 2. Tabla `dim_clientes`
Clientes con historial de segmento (SCD2): una fila por versión, con su vigencia.

**Máscaras de datos personales:** el `schema` del decorador declara `MASK` sobre `nombre`, `apellido` y `email_normalizado`. Quien consulta y no está en `grp-andina-pii-lectores` ve `Q***` y `***@mail.com`; el resto de columnas (segmento, ciudad, país, banderas) queda completo para analítica.

In [ ]:
@dlt.table(
    name="dim_clientes",
    comment="Clientes con historial de segmento (SCD2): una fila por versión, con su vigencia. "
            "Nombre, apellido y email enmascarados salvo para grp-andina-pii-lectores.",
    # El esquema declara las máscaras de columna de Unity Catalog. Nombres y tipos deben
    # coincidir con lo que devuelve la función. Las claves no se enmascaran: los hechos cruzan por ellas.
    # codigo_pais es CHAR(2) en la fuente, pero un esquema declarado no acepta CHAR/VARCHAR: va como STRING.
    schema=f"""
        sk_cliente BIGINT,
        id_cliente INT,
        nombre STRING MASK {GOBIERNO}.mask_nombre,
        apellido STRING MASK {GOBIERNO}.mask_nombre,
        email_normalizado STRING MASK {GOBIERNO}.mask_email,
        ciudad STRING,
        codigo_pais STRING,
        segmento STRING,
        fecha_alta DATE,
        es_activo BOOLEAN,
        es_email_valido BOOLEAN,
        es_posible_duplicado BOOLEAN,
        vigente_desde TIMESTAMP,
        vigente_hasta TIMESTAMP,
        es_vigente BOOLEAN
    """)
def dim_clientes():
    historial = leer_silver("clientes_historial")
    actual = leer_silver("clientes").select(
        "id_cliente", "nombre", "apellido", "email_normalizado", "email_canonico", "telefono", "ciudad",
        "codigo_pais", "fecha_alta", "es_activo", "es_email_valido")

    primera = Window.partitionBy("id_cliente").orderBy(F.col("__START_AT._fecha_efectiva"))
    versiones = (historial
                 .withColumn("_orden", F.row_number().over(primera))
                 .select("id_cliente", "segmento",
                         F.when(F.col("_orden") == 1, INICIO_DE_LOS_TIEMPOS)
                          .otherwise(F.col("__START_AT._fecha_efectiva")).alias("vigente_desde"),
                         F.coalesce(F.col("__END_AT._fecha_efectiva"), FIN_DE_LOS_TIEMPOS).alias("vigente_hasta"),
                         F.col("__END_AT").isNull().alias("es_vigente")))

    por_email = Window.partitionBy("email_canonico")
    por_persona = Window.partitionBy("nombre", "apellido", "telefono")
    actual = actual.withColumn(
        "es_posible_duplicado",
        (F.size(F.collect_set("id_cliente").over(por_email)) > 1)
        | (F.col("telefono").isNotNull() & (F.size(F.collect_set("id_cliente").over(por_persona)) > 1)))

    return (versiones.join(actual, "id_cliente", "inner")
            .select(sk("id_cliente", "vigente_desde").alias("sk_cliente"), "id_cliente", "nombre", "apellido",
                    "email_normalizado", "ciudad", "codigo_pais", "segmento", "fecha_alta", "es_activo",
                    "es_email_valido", "es_posible_duplicado", "vigente_desde", "vigente_hasta", "es_vigente"))